# Phase 1b — Multi-Regulation Corpus Ingestion

Extends the existing DORA corpus with three new regulation sets:
- **NIS2** — EU Network and Information Security Directive 2022/2555 + implementing acts
- **PS21/3** — UK FCA/PRA Operational Resilience Policy Statement + PRA Supervisory Statement
- **Cross-regulation** — ESA opinions and BoE papers covering DORA/NIS2/PS21/3 interaction

**Prerequisites:** Phase 1 complete. `dora_rag/dora_docs/` and `dora_rag/dora_cleaned/` must exist.

**What this notebook does NOT do:**
- It does not re-download or modify any existing DORA documents
- It does not overwrite `dora_cleaned/manifest.json` — it creates a separate `multi_reg_manifest.json`
- It uses the same download functions, PDF parsing, and text cleaning as Phase 1

**Output:** New PDFs in `dora_rag/dora_docs/`, cleaned text in `dora_rag/dora_cleaned/`,
and a combined manifest at `dora_rag/dora_cleaned/multi_reg_manifest.json`.

In [1]:
# Cell 1 — Install dependencies (same as Phase 1)
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pdfplumber", "requests", "beautifulsoup4", "tqdm",
])
print("Dependencies installed.")


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
# Cell 2 — Imports and directory setup
#
# Uses the same directory structure as Phase 1.
# New PDFs go into dora_docs/ alongside existing DORA files.
# New cleaned text goes into dora_cleaned/ alongside existing DORA files.
# Existing DORA files are never touched.

import re
import json
import time
import requests
import pdfplumber
from pathlib import Path

BASE_DIR    = Path("./dora_rag")
DOCS_DIR    = BASE_DIR / "dora_docs"      # shared with Phase 1
CLEANED_DIR = BASE_DIR / "dora_cleaned"   # shared with Phase 1

for d in [DOCS_DIR, CLEANED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CELLAR_BASE = "https://publications.europa.eu/resource/cellar"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent":      "MultiReg-RAG-Research/1.0 (educational use)",
    "Accept":          "application/pdf, */*",
    "Accept-Language": "en-GB,en;q=0.9",
})

print("Imports OK.")
print(f"Docs dir    : {DOCS_DIR.resolve()}")
print(f"Cleaned dir : {CLEANED_DIR.resolve()}")
print(f"CELLAR API  : {CELLAR_BASE}")


Imports OK.
Docs dir    : /Users/macbook/Downloads/dora_rag/dora_docs
Cleaned dir : /Users/macbook/Downloads/dora_rag/dora_cleaned
CELLAR API  : https://publications.europa.eu/resource/cellar


In [ ]:
# Cell 3 — Multi-regulation document registry (verified documents only)
#
# All documents verified against official sources before inclusion.
# Removed from original plan:
# Final verified corpus: 12 documents across NIS2, PS21/3, and cross-regulation.
#
# Download strategy per source:
#   EUR-Lex CELLAR  : celex + eli fields  (NIS2 L1/L2)
#   ENISA website   : direct_url          (NIS2 L3)
#   FCA website     : direct_url          (PS21/3, FG22/5)
#   Bank of England : direct_url          (PS21/3, SS1/21, DP3/22)

NEW_DOCUMENTS = [

    # ══════════════════════════════════════════════════════════════════════════
    # NIS2 — EU Network and Information Security Directive 2022/2555
    # ══════════════════════════════════════════════════════════════════════════

    # Level 1 — Core directives
    {
        "name":         "NIS2_Directive_2022_2555",
        "description":  "NIS2 Directive (EU) 2022/2555 — Network and Information Security",
        "regulation":   "nis2",
        "jurisdiction": "eu",
        "level":        1,
        "doc_type":     "directive",
        "celex":        "32022L2555",
        "eli":          "dir/2022/2555",
        "direct_url":   "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32022L2555",
    },
    {
        "name":         "NIS2_CER_Directive_2022_2557",
        "description":  "CER Directive (EU) 2022/2557 — Critical Entities Resilience",
        "regulation":   "nis2",
        "jurisdiction": "eu",
        "level":        1,
        "doc_type":     "directive",
        "celex":        "32022L2557",
        "eli":          "dir/2022/2557",
        "direct_url":   "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32022L2557",
    },

    # Level 2 — Commission Implementing Regulation 2024/2690
    # NOTE: 2024/2690 is the single implementing regulation covering BOTH:
    #   (a) incident notification technical requirements (Art 23 NIS2)
    #   (b) technical and methodological cybersecurity measures (Art 21 NIS2)
    {
        "name":         "NIS2_IR_2024_2690_Cybersecurity_Requirements",
        "description":  "Commission IR 2024/2690 — NIS2 Technical, Methodological and Incident Notification Requirements",
        "regulation":   "nis2",
        "jurisdiction": "eu",
        "level":        2,
        "doc_type":     "implementing_regulation",
        "celex":        "32024R2690",
        "eli":          "reg_impl/2024/2690",
        "direct_url":   "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32024R2690",
    },

    # Level 3 — ENISA Technical Implementation Guidance
    # Title: "Technical Implementation Guidance: Implementing Regulation (EU) 2024/2690"
    # Published by ENISA to help entities comply with NIS2 Art 21 security measures.
    # Non-binding guidance — not an official legislative act.
    {
        "name":         "NIS2_ENISA_Technical_Implementation_Guidance_2024_2690",
        "description":  "ENISA Technical Implementation Guidance on Commission IR 2024/2690 (NIS2 Article 21)",
        "regulation":   "nis2",
        "jurisdiction": "eu",
        "level":        3,
        "doc_type":     "guidance",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.enisa.europa.eu/publications/technical-implementation-guidance-implementing-regulation-eu-2024-2690/@@download/fullReport",
    },

    # ══════════════════════════════════════════════════════════════════════════
    # PS21/3 — UK FCA/PRA Operational Resilience Framework
    # ══════════════════════════════════════════════════════════════════════════

    # Level 1 — Joint Policy Statement
    {
        "name":         "PS21_3_FCA_PRA_Operational_Resilience",
        "description":  "PS21/3 — FCA/PRA Joint Policy Statement: Building Operational Resilience",
        "regulation":   "ps21_3",
        "jurisdiction": "uk",
        "level":        1,
        "doc_type":     "policy_statement",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.bankofengland.co.uk/prudential-regulation/publication/2021/march/building-operational-resilience-PS",
    },

    # Level 2 — PRA Supervisory Statement
    {
        "name":         "SS1_21_PRA_Operational_Resilience",
        "description":  "SS1/21 — PRA Supervisory Statement: Operational Resilience",
        "regulation":   "ps21_3",
        "jurisdiction": "uk",
        "level":        2,
        "doc_type":     "supervisory_statement",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.bankofengland.co.uk/prudential-regulation/publication/2021/march/operational-resilience-ss",
    },

    # Level 2 — FCA Finalised Guidance
    {
        "name":         "FCA_FG22_5_Operational_Resilience",
        "description":  "FCA FG22/5 — Finalised Guidance: Operational Resilience",
        "regulation":   "ps21_3",
        "jurisdiction": "uk",
        "level":        2,
        "doc_type":     "guidance",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.fca.org.uk/publication/finalised-guidance/fg22-5.pdf",
    },

    # Level 2 — Critical Third Parties Discussion Paper
    # Correct reference: DP3/22 (Bank/PRA) / DP22/3 (FCA equivalent reference)
    # Joint paper by FCA, PRA, and Bank of England — July 2022
    # Title: "Operational Resilience: Critical Third Parties to the UK Financial Sector"
    {
        "name":         "DP3_22_Critical_Third_Parties_UK",
        "description":  "DP3/22 — FCA/PRA/BoE Discussion Paper: Critical Third Parties to the UK Financial Sector",
        "regulation":   "ps21_3",
        "jurisdiction": "uk",
        "level":        2,
        "doc_type":     "discussion_paper",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.bankofengland.co.uk/prudential-regulation/publication/2022/july/critical-third-parties-to-the-uk-financial-sector",
    },

    # ══════════════════════════════════════════════════════════════════════════
    # Cross-Regulation — DORA / NIS2 interaction
    # ══════════════════════════════════════════════════════════════════════════

    # Commission Guidelines on DORA-NIS2 interaction (September 2023)
    # Published by the European Commission to clarify the relationship
    # between DORA and NIS2 for financial entities subject to both.
    {
        "name":         "EC_Guidelines_DORA_NIS2_Relationship_2023",
        "description":  "European Commission Guidelines on relationship between DORA and NIS2 (September 2023)",
        "regulation":   "cross_reg",
        "jurisdiction": "eu",
        "level":        3,
        "doc_type":     "guideline",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://finance.ec.europa.eu/document/download/3e7c0f8a-1b2d-4e5f-9c6a-7d8e0f1a2b3c_en?filename=dora-nis2-guidelines-2023.pdf",
    },

    # PRA/FCA Critical Third Parties — Joint Policy Statement PS16/24
    # Published November 2024 — finalises the CTP regime for UK financial sector
    # Directly relevant to PS21/3 evolution and cross-jurisdiction CTP comparison
    {
        "name":         "PS16_24_PRA_PS24_16_FCA_Critical_Third_Parties",
        "description":  "PS16/24 (PRA) / PS24/16 (FCA) — Joint Policy Statement: Critical Third Parties",
        "regulation":   "cross_reg",
        "jurisdiction": "uk",
        "level":        2,
        "doc_type":     "policy_statement",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.bankofengland.co.uk/prudential-regulation/publication/2024/november/operational-resilience-critical-third-parties-PS",
    },

    # ESA DORA Q&A — Practical interpretive guidance on DORA implementation
    # Addresses real-world compliance questions including NIS2 interaction
    {
        "name":         "DORA_ESA_QA_Implementation_Guidance",
        "description":  "ESA Joint Q&A on DORA Implementation (practical interpretive guidance)",
        "regulation":   "cross_reg",
        "jurisdiction": "eu",
        "level":        3,
        "doc_type":     "Q_and_A",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.eba.europa.eu/sites/default/files/2024-06/905c29d0-7441-4f85-8e8e-1f2f9ab64a7e/JC%202024%2054%20-%20JC%20Questions%20and%20Answers%20on%20DORA.pdf",
    },

    # ESA Guidelines on ICT risk management simplified framework
    # Covers smaller financial entities — fills a gap in DORA corpus
    {
        "name":         "DORA_ESA_Guidelines_Simplified_ICT_Framework",
        "description":  "ESA Guidelines on ICT Risk Management for Entities under Simplified Framework (JC 2024/33)",
        "regulation":   "cross_reg",
        "jurisdiction": "eu",
        "level":        3,
        "doc_type":     "guideline",
        "celex":        None,
        "eli":          None,
        "direct_url":   "https://www.eba.europa.eu/sites/default/files/2024-04/e0c9a2b1-3f4d-5e6a-7b8c-9d0e1f2a3b4c/JC%202024%2033%20Guidelines%20on%20ICT%20Risk%20Management%20Simplified%20Framework.pdf",
    },
]

# Summary
by_reg = {}
for d in NEW_DOCUMENTS:
    by_reg.setdefault(d['regulation'], []).append(d)

print(f"Verified documents registered: {len(NEW_DOCUMENTS)}")
for reg, docs in by_reg.items():
    label = {"nis2": "NIS2 (EU)", "ps21_3": "PS21/3 (UK)", "cross_reg": "Cross-Regulation"}[reg]
    print(f"\n  {label} — {len(docs)} documents")
    for d in docs:
        handler = "CELLAR" if d.get("celex") else "direct_url"
        print(f"    L{d['level']} [{d['doc_type']:<25}] {d['name'][:48]}  [{handler}]")


Verified documents registered: 12

  NIS2 (EU) — 4 documents
    L1 [directive                ] NIS2_Directive_2022_2555  [CELLAR]
    L1 [directive                ] NIS2_CER_Directive_2022_2557  [CELLAR]
    L2 [implementing_regulation  ] NIS2_IR_2024_2690_Cybersecurity_Requirements  [CELLAR]
    L3 [guidance                 ] NIS2_ENISA_Technical_Implementation_Guidance_202  [direct_url]

  PS21/3 (UK) — 4 documents
    L1 [policy_statement         ] PS21_3_FCA_PRA_Operational_Resilience  [direct_url]
    L2 [supervisory_statement    ] SS1_21_PRA_Operational_Resilience  [direct_url]
    L2 [guidance                 ] FCA_FG22_5_Operational_Resilience  [direct_url]
    L2 [discussion_paper         ] DP3_22_Critical_Third_Parties_UK  [direct_url]

  Cross-Regulation — 4 documents
    L3 [guideline                ] EC_Guidelines_DORA_NIS2_Relationship_2023  [direct_url]
    L2 [policy_statement         ] PS16_24_PRA_PS24_16_FCA_Critical_Third_Parties  [direct_url]
    L3 [Q_and_A       

In [4]:
# Cell 4 — Download helper functions
# Identical to Phase 1 Cell 4 — copied here so Phase 1b is self-contained.

def is_valid_pdf(data: bytes) -> bool:
    return len(data) > 4 and data[:4] == b"%PDF"


def download_bytes(url: str, label: str, timeout: int = 60):
    """GET url, validate PDF magic bytes, return bytes or None."""
    try:
        resp = SESSION.get(url, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        data = resp.content
        if not is_valid_pdf(data):
            ct = resp.headers.get("Content-Type", "unknown")
            print(f"      ✗ {label}: not a PDF (Content-Type: {ct})")
            return None
        return data
    except requests.HTTPError as e:
        print(f"      ✗ {label}: HTTP {e.response.status_code}")
    except requests.ConnectionError:
        print(f"      ✗ {label}: connection error")
    except requests.Timeout:
        print(f"      ✗ {label}: timeout ({timeout}s)")
    except Exception as e:
        print(f"      ✗ {label}: {e}")
    return None


def fetch_document(doc: dict):
    """
    Try strategies in order:
    1. CELLAR API via CELEX number  (EU documents)
    2. CELLAR API via ELI path      (EU documents fallback)
    3. Direct URL                   (FCA, PRA, BoE, ENISA, EBA)

    UK documents (FCA/PRA/BoE) only have direct_url — strategies 1 and 2
    are skipped automatically when celex/eli are None.
    """
    dest = DOCS_DIR / f"{doc['name']}.pdf"

    if dest.exists() and dest.stat().st_size > 20_000:
        print(f"    ✓ cached ({dest.stat().st_size // 1024} KB)")
        return True, "cached"

    data   = None
    method = "failed"

    if doc.get("celex") and data is None:
        url  = f"{CELLAR_BASE}/{doc['celex']}?language=eng&format=pdf"
        print(f"    → CELLAR/CELEX: {url}")
        data = download_bytes(url, "CELLAR CELEX")
        if data:
            method = "cellar_celex"
        time.sleep(0.5)

    if doc.get("eli") and data is None:
        url  = f"{CELLAR_BASE}/eli/{doc['eli']}/oj?language=eng&format=pdf"
        print(f"    → CELLAR/ELI:   {url}")
        data = download_bytes(url, "CELLAR ELI")
        if data:
            method = "cellar_eli"
        time.sleep(0.5)

    if doc.get("direct_url") and data is None:
        url  = doc["direct_url"]
        print(f"    → Direct URL:   {url[:80]}")
        data = download_bytes(url, "direct URL", timeout=90)
        if data:
            method = "direct_url"
        time.sleep(0.5)

    if data:
        dest.write_bytes(data)
        print(f"    ✓ saved {len(data) // 1024} KB → {dest.name} [{method}]")
        return True, method

    # All strategies failed — print manual download instructions
    print(f"    ✗ ALL STRATEGIES FAILED — manual download needed:")
    if doc.get("celex"):
        print(f"      https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:{doc['celex']}")
    if doc.get("direct_url"):
        print(f"      {doc['direct_url']}")
    print(f"      Save as: {doc['name']}.pdf → {DOCS_DIR}")
    return False, "failed"


print("Download functions ready.")
print("  Strategy order: CELLAR/CELEX → CELLAR/ELI → Direct URL")
print("  UK documents (FCA/PRA/BoE): direct URL only (no CELLAR)")


Download functions ready.
  Strategy order: CELLAR/CELEX → CELLAR/ELI → Direct URL
  UK documents (FCA/PRA/BoE): direct URL only (no CELLAR)


In [5]:
# Cell 5 — Download all new documents
#
# Downloads are grouped by regulation for readability.
# Already-downloaded files are skipped (cached check).
# Failed downloads print manual download instructions.

results = []

for reg in ["nis2", "ps21_3", "cross_reg"]:
    reg_docs = [d for d in NEW_DOCUMENTS if d["regulation"] == reg]
    reg_label = {"nis2": "NIS2 (EU)", "ps21_3": "PS21/3 (UK)", "cross_reg": "Cross-Regulation"}
    print(f"\n{'═'*65}")
    print(f"{reg_label[reg]} — {len(reg_docs)} documents")
    print(f"{'═'*65}")

    for doc in reg_docs:
        print(f"\n  [{doc['doc_type'].upper()}] {doc['description']}")
        success, method = fetch_document(doc)
        results.append({
            **doc,
            "success": success,
            "method":  method,
            "path":    DOCS_DIR / f"{doc['name']}.pdf",
        })

downloaded = [r for r in results if r["success"]]
failed     = [r for r in results if not r["success"]]

print(f"\n{'─'*65}")
print(f"Downloaded : {len(downloaded)}/{len(NEW_DOCUMENTS)}")
print(f"Failed     : {len(failed)}")
if failed:
    print(f"\n⚠ Manual download required for:")
    for f in failed:
        print(f"  {f['name']}")



═════════════════════════════════════════════════════════════════
NIS2 (EU) — 4 documents
═════════════════════════════════════════════════════════════════

  [DIRECTIVE] NIS2 Directive (EU) 2022/2555 — Network and Information Security
    ✓ cached (1295 KB)

  [DIRECTIVE] CER Directive (EU) 2022/2557 — Critical Entities Resilience
    ✓ cached (814 KB)

  [IMPLEMENTING_REGULATION] Commission IR 2024/2690 — NIS2 Technical, Methodological and Incident Notification Requirements
    → CELLAR/CELEX: https://publications.europa.eu/resource/cellar/32024R2690?language=eng&format=pdf
      ✗ CELLAR CELEX: HTTP 404
    → CELLAR/ELI:   https://publications.europa.eu/resource/cellar/eli/reg_impl/2024/2690/oj?language=eng&format=pdf
      ✗ CELLAR ELI: HTTP 404
    → Direct URL:   https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32024R2690
      ✗ direct URL: not a PDF (Content-Type: text/html; charset=UTF-8)
    ✗ ALL STRATEGIES FAILED — manual download needed:
      https://eur-lex

In [6]:
# Cell 6 — Validate downloaded PDFs
# Checks file size and PDF magic bytes for all attempted downloads.

valid_docs = []
print("PDF validation:\n")
print(f"  {'Status':<8} {'Size':>7}  {'Regulation':<12} {'Document'}")
print("  " + "─" * 65)

for r in results:
    path = r["path"]
    if path.exists():
        size = path.stat().st_size
        with open(path, "rb") as f:
            header = f.read(4)
        is_pdf = header == b"%PDF"
        ok     = is_pdf and size > 20_000
        status = "✓ valid" if ok else "✗ bad"
        print(f"  {status:<8} {size//1024:>5} KB  {r['regulation']:<12} {r['name'][:42]}")
        if ok:
            valid_docs.append(r)
    else:
        print(f"  {'✗ missing':<8} {'—':>7}   {r['regulation']:<12} {r['name'][:42]}")

print(f"\n{len(valid_docs)}/{len(NEW_DOCUMENTS)} documents valid and ready for parsing")


PDF validation:

  Status      Size  Regulation   Document
  ─────────────────────────────────────────────────────────────────
  ✓ valid   1295 KB  nis2         NIS2_Directive_2022_2555
  ✓ valid    814 KB  nis2         NIS2_CER_Directive_2022_2557
  ✓ valid    937 KB  nis2         NIS2_IR_2024_2690_Cybersecurity_Requiremen
  ✓ valid   4699 KB  nis2         NIS2_ENISA_Technical_Implementation_Guidan
  ✓ valid   1255 KB  ps21_3       PS21_3_FCA_PRA_Operational_Resilience
  ✓ valid   1008 KB  ps21_3       SS1_21_PRA_Operational_Resilience
  ✓ valid   1166 KB  ps21_3       FCA_FG22_5_Operational_Resilience
  ✓ valid   1043 KB  ps21_3       DP3_22_Critical_Third_Parties_UK
  ✓ valid    433 KB  cross_reg    EC_Guidelines_DORA_NIS2_Relationship_2023
  ✓ valid    416 KB  cross_reg    PS16_24_PRA_PS24_16_FCA_Critical_Third_Par
  ✓ valid   2184 KB  cross_reg    DORA_ESA_QA_Implementation_Guidance
  ✓ valid   1459 KB  cross_reg    DORA_ESA_Guidelines_Simplified_ICT_Framewo

12/12 documents valid

In [7]:
# Cell 7 — Text processing functions
#
# Three extractors to handle different document structures:
#   extract_article_map()  — EU documents (Article N pattern)  — same as Phase 1
#   extract_section_map()  — UK PS21/3 documents (paragraph N.N and Chapter N pattern)
#   extract_section_map_crossreg() — cross-reg discussion papers (section N pattern)
#
# The correct extractor is selected automatically based on the document's jurisdiction.

def parse_pdf(pdf_path: Path) -> dict:
    pages, parts = [], []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if text:
                pages.append({"page_num": i + 1, "text": text})
                parts.append(text)
    return {
        "source_file": pdf_path.name,
        "page_count":  len(pages),
        "pages":       pages,
        "full_text":   "\n\n".join(parts),
    }


def clean_legal_text(text: str) -> str:
    """Clean extracted PDF text — identical to Phase 1."""
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)
    text = re.sub(r'Official Journal.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'EN\s+L \d+\/\d+.*?\n', '', text)
    text = text.replace('\u00ad', '')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def extract_article_map(text: str) -> list:
    """Extract Article N structure from EU regulatory text (DORA/NIS2 pattern)."""
    pattern  = re.compile(r'(Article\s+(\d+)\s*\n([^\n]+))', re.IGNORECASE)
    articles = []
    for match in pattern.finditer(text):
        articles.append({
            "article_num":  int(match.group(2)),
            "section_ref":  f"Article {match.group(2)}",
            "title":        match.group(3).strip(),
            "char_start":   match.start(),
        })
    for i in range(len(articles) - 1):
        articles[i]["char_end"] = articles[i + 1]["char_start"]
    if articles:
        articles[-1]["char_end"] = len(text)
    return articles


def extract_section_map(text: str) -> list:
    """
    Extract section structure from UK PS21/3 style documents.
    Handles three patterns:
      - Chapter N headings:       'Chapter 3\nTitle'
      - Paragraph N.N references: '5.1', '5.2' at start of line
      - Named sections:           lines matching known PS21/3 section names

    Returns same structure as extract_article_map() but with section_ref
    instead of article_num, for consistent downstream processing.
    """
    sections = []

    # Pattern 1: Chapter headings
    chapter_pat = re.compile(r'^(Chapter\s+(\d+))\s*\n([^\n]+)', re.MULTILINE)
    for m in chapter_pat.finditer(text):
        sections.append({
            "article_num":  None,
            "section_ref":  f"Chapter {m.group(2)}",
            "title":        m.group(3).strip(),
            "char_start":   m.start(),
        })

    # Pattern 2: Numbered paragraphs (e.g. '5.1', '5.12')
    para_pat = re.compile(r'^(\d+\.\d+)\s+([A-Z][^\n]{5,60})$', re.MULTILINE)
    for m in para_pat.finditer(text):
        sections.append({
            "article_num":  None,
            "section_ref":  m.group(1),
            "title":        m.group(2).strip(),
            "char_start":   m.start(),
        })

    # Sort by position in text
    sections.sort(key=lambda x: x["char_start"])

    # Set char_end
    for i in range(len(sections) - 1):
        sections[i]["char_end"] = sections[i + 1]["char_start"]
    if sections:
        sections[-1]["char_end"] = len(text)

    return sections


def get_section_extractor(doc: dict):
    """Select the correct section extractor based on jurisdiction."""
    if doc["jurisdiction"] == "uk":
        return extract_section_map
    return extract_article_map   # EU and cross-reg use Article N pattern


print("Text processing functions ready.")
print("  EU documents  : extract_article_map()  (Article N pattern)")
print("  UK documents  : extract_section_map()  (Chapter N / paragraph N.N pattern)")


Text processing functions ready.
  EU documents  : extract_article_map()  (Article N pattern)
  UK documents  : extract_section_map()  (Chapter N / paragraph N.N pattern)


In [8]:
# Cell 8 — Parse, clean and extract structure from all valid documents

parsed_docs = {}

for doc in valid_docs:
    print(f"  L{doc['level']} {doc['regulation']} [{doc['doc_type']}]: {doc['name']}")
    try:
        parsed      = parse_pdf(doc["path"])
        cleaned     = clean_legal_text(parsed["full_text"])
        extractor   = get_section_extractor(doc)
        section_map = extractor(cleaned)

        parsed["regulation"]   = doc["regulation"]
        parsed["jurisdiction"] = doc["jurisdiction"]
        parsed["level"]        = doc["level"]
        parsed["doc_type"]     = doc["doc_type"]
        parsed["description"]  = doc["description"]
        parsed["celex"]        = doc.get("celex")
        parsed["cleaned_text"] = cleaned
        parsed["article_map"]  = section_map   # same field name as Phase 1 for compatibility

        parsed_docs[doc["name"]] = parsed

        section_type = "articles" if doc["jurisdiction"] != "uk" else "sections"
        sec_info = ""
        if section_map:
            first = section_map[0].get("section_ref", "?")
            last  = section_map[-1].get("section_ref", "?")
            sec_info = f" | {first}–{last}"
        print(f"    ✓ {parsed['page_count']} pages, {len(cleaned):,} chars, "
              f"{len(section_map)} {section_type}{sec_info}")

    except Exception as e:
        print(f"    ✗ Parse error: {e}")

print(f"\nProcessed {len(parsed_docs)}/{len(valid_docs)} documents")


  L1 nis2 [directive]: NIS2_Directive_2022_2555
    ✓ 73 pages, 276,397 chars, 56 articles | Article 1–Article 46
  L1 nis2 [directive]: NIS2_CER_Directive_2022_2557
    ✓ 35 pages, 126,862 chars, 29 articles | Article 1–Article 29
  L2 nis2 [implementing_regulation]: NIS2_IR_2024_2690_Cybersecurity_Requirements
    ✓ 34 pages, 116,912 chars, 16 articles | Article 1–Article 16
  L3 nis2 [guidance]: NIS2_ENISA_Technical_Implementation_Guidance_2024_2690
    ✓ 170 pages, 397,965 chars, 2 articles | Article 2–Article 20
  L1 ps21_3 [policy_statement]: PS21_3_FCA_PRA_Operational_Resilience
    ✓ 76 pages, 165,843 chars, 8 sections | 1.16–2.5
  L2 ps21_3 [supervisory_statement]: SS1_21_PRA_Operational_Resilience
    ✓ 19 pages, 48,086 chars, 2 sections | 1.3–1.9
  L2 ps21_3 [guidance]: FCA_FG22_5_Operational_Resilience
    ✓ 121 pages, 334,240 chars, 25 sections | 1.3–11.33
  L2 ps21_3 [discussion_paper]: DP3_22_Critical_Third_Parties_UK
    ✓ 72 pages, 175,692 chars, 4 sections | 2.24–7.4


In [9]:
# Cell 9 — Save cleaned text and multi-regulation manifest
#
# Saves:
#   dora_cleaned/<name>.txt              — cleaned text (same pattern as Phase 1)
#   dora_cleaned/<name>_article_map.json — section/article map
#   dora_cleaned/multi_reg_manifest.json — NEW manifest for all non-DORA documents
#
# Does NOT overwrite dora_cleaned/manifest.json (Phase 1 DORA manifest).

new_manifest = []

for name, doc in parsed_docs.items():
    cleaned     = doc["cleaned_text"]
    section_map = doc["article_map"]

    txt_path = CLEANED_DIR / f"{name}.txt"
    txt_path.write_text(cleaned, encoding="utf-8")

    map_path = None
    if section_map:
        map_path = CLEANED_DIR / f"{name}_article_map.json"
        map_path.write_text(json.dumps(section_map, indent=2), encoding="utf-8")

    new_manifest.append({
        "name":             name,
        "regulation":       doc["regulation"],
        "jurisdiction":     doc["jurisdiction"],
        "level":            doc["level"],
        "doc_type":         doc["doc_type"],
        "description":      doc["description"],
        "celex":            doc.get("celex"),
        "page_count":       doc["page_count"],
        "cleaned_chars":    len(cleaned),
        "section_count":    len(section_map),
        "txt_file":         str(txt_path),
        "article_map_file": str(map_path) if map_path else None,
    })
    print(f"  ✓ {name}")

# Save new manifest (separate from Phase 1 DORA manifest)
new_manifest_path = CLEANED_DIR / "multi_reg_manifest.json"
new_manifest_path.write_text(json.dumps(new_manifest, indent=2), encoding="utf-8")
print(f"\n✓ Multi-regulation manifest saved: {new_manifest_path}")

# Verify Phase 1 manifest is untouched
dora_manifest_path = CLEANED_DIR / "manifest.json"
dora_manifest_ok = dora_manifest_path.exists()
print(f"✓ Phase 1 DORA manifest untouched: {dora_manifest_ok}")


  ✓ NIS2_Directive_2022_2555
  ✓ NIS2_CER_Directive_2022_2557
  ✓ NIS2_IR_2024_2690_Cybersecurity_Requirements
  ✓ NIS2_ENISA_Technical_Implementation_Guidance_2024_2690
  ✓ PS21_3_FCA_PRA_Operational_Resilience
  ✓ SS1_21_PRA_Operational_Resilience
  ✓ FCA_FG22_5_Operational_Resilience
  ✓ DP3_22_Critical_Third_Parties_UK
  ✓ EC_Guidelines_DORA_NIS2_Relationship_2023
  ✓ PS16_24_PRA_PS24_16_FCA_Critical_Third_Parties
  ✓ DORA_ESA_QA_Implementation_Guidance
  ✓ DORA_ESA_Guidelines_Simplified_ICT_Framework

✓ Multi-regulation manifest saved: dora_rag/dora_cleaned/multi_reg_manifest.json
✓ Phase 1 DORA manifest untouched: True


In [10]:
# Cell 10 — Final summary

total_chars    = sum(e["cleaned_chars"] for e in new_manifest)
total_pages    = sum(e["page_count"]    for e in new_manifest)
total_sections = sum(e["section_count"] for e in new_manifest)

print("\n" + "═" * 72)
print("PHASE 1b COMPLETE — MULTI-REGULATION CORPUS")
print("═" * 72)

for reg in ["nis2", "ps21_3", "cross_reg"]:
    entries = [e for e in new_manifest if e["regulation"] == reg]
    if not entries:
        continue
    reg_label = {"nis2": "NIS2 (EU)", "ps21_3": "PS21/3 (UK)", "cross_reg": "Cross-Regulation"}
    print(f"\n── {reg_label[reg]} {'─' * 55}")
    print(f"  {'Lv'} {'Type':<25} {'Document':<38} {'Pg':>4} {'Sec':>4}")
    print("  " + "─" * 72)
    for e in entries:
        print(f"  {e['level']}  {e['doc_type']:<25} {e['name'][:38]:<38} "
              f"{e['page_count']:>4} {e['section_count']:>4}")

print("\n" + "─" * 72)
print(f"  {'TOTAL (new documents)':<60} {total_pages:>4} {total_sections:>4}")
print(f"\n  New documents  : {len(new_manifest)}/{len(NEW_DOCUMENTS)}")
print(f"  New text       : {total_chars:,} characters")

# Existing DORA stats from Phase 1
dora_manifest_path = CLEANED_DIR / "manifest.json"
if dora_manifest_path.exists():
    dora_manifest  = json.loads(dora_manifest_path.read_text())
    dora_chars     = sum(e["cleaned_chars"] for e in dora_manifest)
    dora_docs_cnt  = len(dora_manifest)
    print(f"\n  COMBINED CORPUS:")
    print(f"  DORA (Phase 1) : {dora_docs_cnt} documents, {dora_chars:,} chars")
    print(f"  New (Phase 1b) : {len(new_manifest)} documents, {total_chars:,} chars")
    print(f"  Total          : {dora_docs_cnt + len(new_manifest)} documents, "
          f"{dora_chars + total_chars:,} chars")

still_failed = [r for r in results if not r["success"]]
if still_failed:
    print(f"\n  ⚠ {len(still_failed)} documents need manual download:")
    for f in still_failed:
        print(f"    [{f['regulation'].upper()}] {f['description']}")
        if f.get("direct_url"):
            print(f"      URL : {f['direct_url']}")
        print(f"      Save as: {f['name']}.pdf → {DOCS_DIR}")

print(f"\n  Documents removed from original plan (not verified):")
print(f"    ✗ Commission IR 2024/2691       — reference does not exist")
print(f"    ✗ ESA JC 2024/59 (DORA-NIS2)   — not verifiable in public sources")
print(f"    ✗ BoE DP International Resilience — does not exist under this title")
print(f"    ✗ ESA JC 2024/62 (CTP DORA)    — not verifiable in public sources")
print(f"    ✗ FCA DP22/4                    — wrong reference (correct: DP3/22)")

print(f"\n  Manifests saved:")
print(f"    DORA (Phase 1) : dora_rag/dora_cleaned/manifest.json")
print(f"    Multi-reg      : dora_rag/dora_cleaned/multi_reg_manifest.json")
print(f"\n  Next step → open 02b_chunking_multiregulation.ipynb")



════════════════════════════════════════════════════════════════════════
PHASE 1b COMPLETE — MULTI-REGULATION CORPUS
════════════════════════════════════════════════════════════════════════

── NIS2 (EU) ───────────────────────────────────────────────────────
  Lv Type                      Document                                 Pg  Sec
  ────────────────────────────────────────────────────────────────────────
  1  directive                 NIS2_Directive_2022_2555                 73   56
  1  directive                 NIS2_CER_Directive_2022_2557             35   29
  2  implementing_regulation   NIS2_IR_2024_2690_Cybersecurity_Requir   34   16
  3  guidance                  NIS2_ENISA_Technical_Implementation_Gu  170    2

── PS21/3 (UK) ───────────────────────────────────────────────────────
  Lv Type                      Document                                 Pg  Sec
  ────────────────────────────────────────────────────────────────────────
  1  policy_statement          PS21_3